<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 03. GraphSAGE: 보지 못한 노드도 임베딩하기

## Inductive Representation Learning on Large Graphs

- **원 논문:** [Inductive Representation Learning on Large Graphs](https://arxiv.org/abs/1706.02216)
- **저자 / 발표:** William L. Hamilton, Rex Ying, Jure Leskovec · NeurIPS (2017)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** GCN, permutation-invariant aggregation
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 고정 크기 이웃 표본과 mean aggregator를 구현하고 feature 기반 inductive 출력을 확인합니다.

**축소하거나 생략한 부분:** Reddit/PPI의 대규모 minibatch sampling 대신 전체 18노드와 작은 fanout을 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, Neighborhood definition | 고정 fanout 이웃 표본 | 개수·seed·edge 유효성 |
| §3.3, Mean aggregator | mean neighborhood aggregation | 순열 불변성 |
| §3.1, Algorithm 1 lines 4–7 | self와 neighbor concat·정규화 | row norm |
| §2 및 §3.1 | unseen-node inductive inference | feature 변경에 따른 출력 |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** sampled A [N,N], X [N,D] → aggregated hidden [N,H]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$h_v^k=\sigma\!\left(W^k[h_v^{k-1}\Vert \operatorname{MEAN}_{u\in\mathcal N(v)}h_u^{k-1}]\right)$$

- **기호 정의:** v는 중심 노드, N(v)는 표본 이웃, ||는 concat, W는 공유 가중치입니다.
- **수식의 역할:** 특징 기반 mean aggregation으로 학습 때 보지 못한 노드도 계산합니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** Task P의 `preprocess_graph`, 논문 전용 클래스의
  `forward`, `compute_loss`, `training_step`이 §3.1, Neighborhood definition을 구조화합니다.
- **입출력 shape:** 특징 [N, F]와 fanout 이웃 → 정규화 표현 [N, H] → logit [N, C]
- **평가:** 분류 정확도와 입력 특징 변화에 대한 inductive 출력 변화
- **Inductive split:** optimizer는 train-induced subgraph만 사용하고,
  held-out 노드의 feature·neighbor·label은 학습 종료 뒤 추론에만 넣습니다.
- **원문 대비 한계:** Reddit/PPI의 대규모 minibatch sampling 대신 전체 18노드와 작은 fanout을 사용합니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1, Neighborhood definition
- **이 셀의 책임:** 고정 fanout 이웃 표본
- **Tensor shape 계약:** sampled A [N,N], X [N,D] → aggregated hidden [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 개수·seed·edge 유효성. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def sample_neighbors(adj: Tensor, fanout: int, seed: int) -> list[Tensor]:
    rng = np.random.default_rng(seed)
    result = []
    for u in range(len(adj)):
        nbr = torch.where(adj[u] > 0)[0].numpy()
        chosen = (
            rng.choice(nbr, min(fanout, len(nbr)), replace=False)
            if len(nbr)
            else np.array([], dtype=int)
        )
        result.append(torch.tensor(chosen, dtype=torch.long))
    return result


sampled = sample_neighbors(adjacency, 3, 13)
assert all(len(v) <= 3 for v in sampled)
assert all(adjacency[u, v].all() for u, v in enumerate(sampled) if len(v))

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.3, Mean aggregator / §3.1, Algorithm 1 lines 4–7
- **이 셀의 책임:** mean neighborhood aggregation / self와 neighbor concat·정규화
- **Tensor shape 계약:** sampled A [N,N], X [N,D] → aggregated hidden [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 순열 불변성 / row norm. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class MeanSAGE(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.linear = nn.Linear(2 * in_dim, out_dim)

    def forward(self, x: Tensor, neighbors: list[Tensor]) -> Tensor:
        agg = torch.stack(
            [
                x[idx].mean(0) if len(idx) else torch.zeros_like(x[0])
                for idx in neighbors
            ]
        )
        return F.normalize(F.relu(self.linear(torch.cat([x, agg], dim=1))), dim=1)


sage = MeanSAGE(features.shape[1], 10)
h = sage(features, sampled)
assert h.shape == (len(features), 10) and torch.allclose(
    h.norm(dim=1)[h.norm(dim=1) > 0],
    torch.ones_like(h.norm(dim=1)[h.norm(dim=1) > 0]),
    atol=1e-5,
)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §2 및 §3.1
- **이 셀의 책임:** unseen-node inductive inference
- **Tensor shape 계약:** sampled A [N,N], X [N,D] → aggregated hidden [N,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** feature 변경에 따른 출력. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
train_index = torch.where(train_mask)[0]
test_index = torch.where(test_mask)[0]
assert not torch.isin(train_index, test_index).any()
train_adjacency = adjacency[train_index][:, train_index]
train_features = features[train_index].clone()
train_labels = labels[train_index].clone()
train_sampled = sample_neighbors(train_adjacency, 3, 13)
classifier = nn.Linear(h.shape[1], int(train_labels.max()) + 1)
opt = torch.optim.Adam([*sage.parameters(), *classifier.parameters()], lr=0.04)
losses = []
for _ in range(140):
    h = sage(train_features, train_sampled)
    logits = classifier(h)
    loss = F.cross_entropy(logits, train_labels)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(float(loss))
with torch.no_grad():
    # New nodes enter only here, with their own features and sampled neighbors.
    base = sage(features, sampled)
    unseen_logits = classifier(base)[test_index]
    changed_features = features.clone()
    changed_features[test_index] += 0.25
    changed = sage(changed_features, sampled)
    sensitivity = float((base[test_index] - changed[test_index]).norm(dim=1).mean())
    test_acc = float((unseen_logits.argmax(1) == labels[test_index]).float().mean())
assert losses[-1] < losses[0] and sensitivity > 0
assert len(train_features) == len(train_labels) == len(train_sampled)
assert train_adjacency.shape == (len(train_index), len(train_index))
plt.plot(losses)
plt.title(f"GraphSAGE · unseen sensitivity={sensitivity:.3f}")
plt.show()
print({"test_accuracy": test_acc, "feature_sensitivity": sensitivity})

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §3.1, Neighborhood definition / §3.3, Mean aggregator / §3.1, Algorithm 1 lines 4–7 / §2 및 §3.1
- **이 셀의 책임:** 고정 fanout 이웃 표본 / mean neighborhood aggregation / self와 neighbor concat·정규화 / unseen-node inductive inference
- **Tensor shape 계약:** sampled A [N,N], X [N,D] → aggregated hidden [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 개수·seed·edge 유효성 / 순열 불변성 / row norm / feature 변경에 따른 출력. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 12
    fanout: int = 3
    sample_seed: int = 13
    learning_rate: float = 0.03
    steps: int = 18


def preprocess_graph(
    graph: Tensor,
    config: PortfolioConfig,
) -> list[Tensor]:
    return sample_neighbors(graph, config.fanout, config.sample_seed)


class GraphsagePortfolioModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        class_count: int,
        config: PortfolioConfig,
    ):
        super().__init__()
        self.config = config
        self.encoder = MeanSAGE(input_dim, config.hidden_dim)
        self.classifier = nn.Linear(config.hidden_dim, class_count)

    def encode(self, inputs: Tensor, neighbors: list[Tensor]) -> Tensor:
        return self.encoder(inputs, neighbors)

    def forward(self, inputs: Tensor, neighbors: list[Tensor]) -> Tensor:
        return self.classifier(self.encode(inputs, neighbors))

    def compute_loss(
        self,
        outputs: Tensor,
        targets: Tensor,
    ) -> Tensor:
        return F.cross_entropy(outputs, targets)

    def training_step(
        self,
        inputs: Tensor,
        neighbors: list[Tensor],
        targets: Tensor,
    ) -> Tensor:
        outputs = self(inputs, neighbors)
        return self.compute_loss(outputs, targets)

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §3.3, Mean aggregator / §3.1, Algorithm 1 lines 4–7
- **이 셀의 책임:** mean neighborhood aggregation / self와 neighbor concat·정규화
- **Tensor shape 계약:** sampled A [N,N], X [N,D] → aggregated hidden [N,H]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 순열 불변성 / row norm. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: GraphsagePortfolioModel,
    inputs: Tensor,
    neighbors: list[Tensor],
    targets: Tensor,
) -> list[float]:
    assert len(inputs) == len(neighbors) == len(targets)
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    model.train()
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, neighbors, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §2 및 §3.1
- **이 셀의 책임:** unseen-node inductive inference
- **Tensor shape 계약:** sampled A [N,N], X [N,D] → aggregated hidden [N,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** feature 변경에 따른 출력. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: GraphsagePortfolioModel,
    inputs: Tensor,
    neighbors: list[Tensor],
    unseen_targets: Tensor,
    unseen_index: Tensor,
) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        base_logits = model(inputs, neighbors)
        unseen_logits = base_logits[unseen_index]
        prediction = unseen_logits.argmax(dim=1)
        accuracy = (prediction == unseen_targets).float().mean()
        changed_inputs = inputs.clone()
        changed_inputs[unseen_index] += 0.25
        changed_logits = model(changed_inputs, neighbors)
        sensitivity = (unseen_logits - changed_logits[unseen_index]).norm(dim=1).mean()
    return {
        "unseen_test_accuracy": float(accuracy.cpu()),
        "unseen_feature_sensitivity": float(sensitivity.cpu()),
    }


portfolio_train_index = torch.where(train_mask)[0]
portfolio_test_index = torch.where(test_mask)[0]
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_graph = adjacency[portfolio_train_index][:, portfolio_train_index]
portfolio_train_features = features[portfolio_train_index].clone()
portfolio_train_labels = labels[portfolio_train_index].clone()
portfolio_inference_features = features.clone()
portfolio_config = PortfolioConfig()
portfolio_train_neighbors = preprocess_graph(
    portfolio_train_graph,
    portfolio_config,
)
portfolio_inference_neighbors = preprocess_graph(
    adjacency,
    portfolio_config,
)
(
    portfolio_train_features,
    portfolio_train_labels,
    portfolio_train_neighbors,
    portfolio_inference_features,
    portfolio_inference_neighbors,
    portfolio_unseen_labels,
    portfolio_test_index_device,
) = ACCELERATOR.move(
    portfolio_train_features,
    portfolio_train_labels,
    portfolio_train_neighbors,
    portfolio_inference_features,
    portfolio_inference_neighbors,
    labels[portfolio_test_index].clone(),
    portfolio_test_index,
)
portfolio_model = ACCELERATOR.move(
    GraphsagePortfolioModel(
        portfolio_train_features.shape[1],
        int(portfolio_train_labels.max()) + 1,
        portfolio_config,
    )
)
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_train_features,
    portfolio_train_neighbors,
    portfolio_train_labels,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_inference_features,
    portfolio_inference_neighbors,
    portfolio_unseen_labels,
    portfolio_test_index_device,
)
assert np.isfinite(portfolio_history).all()
assert 0.0 <= portfolio_metrics["unseen_test_accuracy"] <= 1.0
assert portfolio_metrics["unseen_feature_sensitivity"] > 0.0
assert len(portfolio_train_features) == len(portfolio_train_index)
assert len(portfolio_train_neighbors) == len(portfolio_train_index)
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> Reddit/PPI의 대규모 minibatch sampling 대신 전체 18노드와 작은 fanout을 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.